In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import time

# checking that we are using a GPU
device = 'gpu:0' if tf.test.is_gpu_available() else 'cpu'
print('using', device, 'device \n')

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
using cpu device 



In [2]:
from cosmosis.samplers.rose.nn_emulator import NNEmulator

Using CPU for computations


In [3]:
import os
dirname = os.getcwd()
print(dirname)
if os.getcwd().endswith("cosmosis-standard-library" + os.path.sep + "examples_rose" + os.path.sep + "notebooks"):
    print("Switching to directory cosmosis-standard-library")
    os.chdir(os.path.pardir)
    os.chdir(os.path.pardir)
dirname = os.getcwd()
print(dirname)

/Users/s2265800/Desktop/Work/cosmosis_stuff/cosmosis-standard-library/examples_rose/notebooks
Switching to directory cosmosis-standard-library
/Users/s2265800/Desktop/Work/cosmosis_stuff/cosmosis-standard-library


In [4]:
emu_info_file = 'output/rose_lssty1/rose_outputs/emumodel_9/lsst'
emu_info = np.load('output/rose_lssty1/rose_outputs/emumodel_9/lsst.npz', allow_pickle=True)
emu = NNEmulator(model_parameters=emu_info['parameters'],
           modes=np.arange(0, len(emu_info['features_mean']), 1),
           nn_model=emu_info['architecture_type'],
           loss_function=emu_info['loss_function'],
           iteration=5,
           data_transformation=emu_info['data_transformation'])
emu.load(emu_info_file)
print(f"Data transformation: {emu.data_transformation}")

Data transformation: signed_log_norm


In [7]:
inv_covariance = np.genfromtxt('output/lssty1_test/data_vector/lsst_inverse_covariance.txt')
print(f"Covariance shape: {inv_covariance.shape}") 
data_vector = np.genfromtxt('output/lssty1_test/data_vector/lsst_data.txt')
print(f"Data vector shape: {data_vector.shape}")

Covariance shape: (489, 489)
Data vector shape: (489,)


In [ ]:
# ---------------------------------------------------------------------------
# Likelihood / chi2 (matches ROSE NUTS Gaussian likelihood)
# ---------------------------------------------------------------------------
# logL = -0.5 * (d - t)^T C^{-1} (d - t)
# Best fit: maximize logL  <=>  minimize chi2  <=>  grad(logL) = 0

SIGNED_LOG_S = 1e-7  # same constant as cosmosis.samplers.rose.sampling

param_names = list(emu_info['parameters'])
n_params = len(param_names)

def params_dict_to_array(params_dict):
    return np.array([float(np.asarray(params_dict[p]).reshape(-1)[0]) for p in param_names], dtype=np.float64)

def params_array_to_dict(theta):
    return {p: float(theta[i]) for i, p in enumerate(param_names)}

def chi2_numpy(params_dict):
    theory = emu.predict(params_dict)[0]
    diff = data_vector - theory
    return float(np.einsum('i,ij,j->', diff, inv_covariance, diff))

def likelihood(params_dict):
    """Gaussian log-likelihood = -0.5 * chi2 (no prior)."""
    return -0.5 * chi2_numpy(params_dict)

# TensorFlow constants for differentiable logL (as in sampling._log_prob_nuts_impl)
DTYPE = tf.float32
X_mean_tf = tf.constant([emu.X_mean[p] for p in param_names], dtype=DTYPE)
X_std_tf = tf.constant([emu.X_std[p] for p in param_names], dtype=DTYPE)
y_mean_tf = tf.constant(emu.y_mean, dtype=DTYPE)
y_std_tf = tf.constant(emu.y_std, dtype=DTYPE)
data_vector_tf = tf.constant(data_vector, dtype=DTYPE)
inv_covariance_tf = tf.constant(inv_covariance, dtype=DTYPE)

def theory_tf(theta_tf):
    """Differentiable emulator forward pass (physical params -> theory vector)."""
    if len(theta_tf.shape) > 1:
        theta_tf = tf.reshape(theta_tf, [-1])
    params_norm = (theta_tf - X_mean_tf) / X_std_tf
    x = (params_norm - emu.cp_nn.parameters_mean) / emu.cp_nn.parameters_std
    x = tf.expand_dims(x, 0)
    layers = [x]
    for i in range(emu.cp_nn.n_layers - 1):
        linear_out = tf.matmul(layers[-1], emu.cp_nn.W[i]) + emu.cp_nn.b[i]
        layers.append(emu.cp_nn.activation(linear_out, emu.cp_nn.alphas[i], emu.cp_nn.betas[i]))
    output = tf.matmul(layers[-1], emu.cp_nn.W[-1]) + emu.cp_nn.b[-1]
    pred_norm = (output * emu.cp_nn.features_std + emu.cp_nn.features_mean)[0]
    pred_intermediate = pred_norm * y_std_tf + y_mean_tf
    if emu.data_transformation == 'log_norm':
        return tf.pow(10.0, pred_intermediate)
    if emu.data_transformation == 'signed_log_norm':
        return tf.sign(pred_intermediate) * SIGNED_LOG_S * (tf.pow(10.0, tf.abs(pred_intermediate)) - 1.0)
    return pred_intermediate

def chi2_tf(theta_tf):
    theory = theory_tf(theta_tf)
    diff = data_vector_tf - theory
    return tf.einsum('i,ij,j->', diff, inv_covariance_tf, diff)

def loglike_tf(theta_tf):
    return -0.5 * chi2_tf(theta_tf)

def loglike_and_grad(theta_np):
    """Return (logL, grad_theta logL) via TF autodiff."""
    theta_tf = tf.Variable(tf.constant(theta_np, dtype=DTYPE))
    with tf.GradientTape() as tape:
        logL = loglike_tf(theta_tf)
    grad = tape.gradient(logL, theta_tf)
    return float(logL.numpy()), grad.numpy().astype(np.float64)

def chi2_and_grad(theta_np):
    """Return (chi2, grad_theta chi2) for minimizers (grad chi2 = -2 grad logL)."""
    logL, g_logL = loglike_and_grad(theta_np)
    return -2.0 * logL, -2.0 * g_logL


In [ ]:
import yaml
import configparser

with open("examples_rose/notebooks/fiducials_lsst_w0wadesi_full.yaml", "r") as file:
    fiducial_dic = yaml.safe_load(file)
    fiducials = {k: v['p0'] for k, v in fiducial_dic.items()}
    params_dic = {k: v['latex'] for k, v in fiducial_dic.items()}
    ranges = {k: v['range'] for k, v in fiducial_dic.items()}

# Optional Gaussian priors (same file as CosmoSIS / Fisher)
priors_cfg = configparser.ConfigParser()
priors_cfg.read('examples_rose/lssty1/lsst_priors_y1.ini')
gaussian_priors = {}  # index -> (mu, sigma)
for i, pname in enumerate(param_names):
    sec, key = pname.split('--', 1)
    if priors_cfg.has_section(sec) and priors_cfg.has_option(sec, key):
        parts = priors_cfg.get(sec, key).split()
        if parts[0].lower() == 'gaussian':
            gaussian_priors[i] = (float(parts[1]), float(parts[2]))

bounds = np.array([ranges[p] for p in param_names], dtype=np.float64)  # (n, 2)

def logpost_and_grad(theta_np, include_gauss_prior=True):
    """log-posterior = logL + log-prior (Gaussian only; flat priors add a constant)."""
    logL, g = loglike_and_grad(theta_np)
    if include_gauss_prior:
        for i, (mu, sigma) in gaussian_priors.items():
            logL += -0.5 * ((theta_np[i] - mu) / sigma)**2
            g[i] += -(theta_np[i] - mu) / sigma**2
    return logL, g

# Smoke test at fiducial
theta0 = params_dict_to_array({p: fiducials[p] for p in param_names})
chi2_0, g_chi2_0 = chi2_and_grad(theta0)
logL_0, g_logL_0 = loglike_and_grad(theta0)
print(f"At fiducial: chi2={chi2_0:.4f}, logL={logL_0:.4f}, ||grad logL||={np.linalg.norm(g_logL_0):.4e}")
print(f"numpy likelihood check: {likelihood({p: fiducials[p] for p in param_names}):.4f}")


In [34]:
# ---------------------------------------------------------------------------
# TensorFlow L-BFGS in *normalized* parameter space
# ---------------------------------------------------------------------------
# Physical θ has wildly different scales (Ωb ~ 1e-3 vs b_i ~ 1), so ||∇|| in
# θ-space is huge even near the minimum and L-BFGS line search fails.
# Optimise x = (θ - X_mean) / X_std (emulator normalisation), then map back.
# Starts: fiducial + small noise in x-space (random draws in the prior box
# extrapolate the NN and produce NaN/Inf).

import tensorflow_probability as tfp

x_mean_np = np.array([emu.X_mean[p] for p in param_names], dtype=np.float64)
x_std_np = np.array([emu.X_std[p] for p in param_names], dtype=np.float64)

def theta_from_x(x_tf):
    return X_mean_tf + X_std_tf * x_tf

def x_from_theta(theta_np):
    return ((np.asarray(theta_np, dtype=np.float64) - x_mean_np) / x_std_np).astype(np.float64)

def project_theta_np(theta_np):
    return np.clip(theta_np, bounds[:, 0], bounds[:, 1])

def neg_logpost_from_x(x_tf, include_gauss_prior=True):
    """Objective in normalised coords: -logpost(θ(x))."""
    theta = theta_from_x(x_tf)
    # Bound penalty (quadratic outside prior box) — keeps L-BFGS from wandering
    # into emulator-extrapolation regions without a hard discontinuous clip.
    below = tf.nn.relu(bounds_lo_tf - theta)
    above = tf.nn.relu(theta - bounds_hi_tf)
    penalty = 1e6 * tf.reduce_sum(tf.square(below) + tf.square(above))

    loss = 0.5 * chi2_tf(theta)
    if include_gauss_prior:
        for i, (mu, sigma) in gaussian_priors.items():
            loss = loss + 0.5 * tf.square((theta[i] - mu) / sigma)
    return loss + penalty

bounds_lo_tf = tf.constant(bounds[:, 0], dtype=DTYPE)
bounds_hi_tf = tf.constant(bounds[:, 1], dtype=DTYPE)


def make_lbfgs_value_and_grad(include_gauss_prior=True):
    def value_and_gradients(x):
        x = tf.convert_to_tensor(x, dtype=DTYPE)
        loss, grad = tfp.math.value_and_gradient(
            lambda z: neg_logpost_from_x(z, include_gauss_prior=include_gauss_prior),
            x,
        )
        # Guard against NaN gradients (line search then fails immediately)
        grad = tf.where(tf.math.is_finite(grad), grad, tf.zeros_like(grad))
        loss = tf.where(tf.math.is_finite(loss), loss, tf.constant(1e20, dtype=DTYPE))
        return loss, grad
    return value_and_gradients


def lbfgs_minimise(theta_start, include_gauss_prior=True,
                   max_iterations=1024, tolerance=1e-4):
    theta_start = project_theta_np(theta_start)
    x0 = tf.constant(x_from_theta(theta_start), dtype=DTYPE)
    vg = make_lbfgs_value_and_grad(include_gauss_prior=include_gauss_prior)

    result = tfp.optimizer.lbfgs_minimize(
        value_and_gradients_function=vg,
        initial_position=x0,
        tolerance=tolerance,
        x_tolerance=1e-5,
        f_relative_tolerance=1e-5,
        max_iterations=max_iterations,
        max_line_search_iterations=126,
        parallel_iterations=1,
    )

    theta_bf = project_theta_np(theta_from_x(result.position).numpy().astype(np.float64))
    return {
        'theta': theta_bf,
        'converged': bool(result.converged.numpy()),
        'failed': bool(result.failed.numpy()),
        'nit': int(result.num_iterations.numpy()),
        'objective_evals': int(result.num_objective_evaluations.numpy()),
        'objective': float(result.objective_value.numpy()),
        'grad_x_norm': float(np.linalg.norm(result.objective_gradient.numpy())),
    }


# Starts near fiducial in normalised space
rng = np.random.default_rng(42)
n_starts = 8
x_fid = x_from_theta(theta0)
starts = []
for _ in range(n_starts):
    #x_pert = x_fid + 0.1 * rng.normal(size=n_params)  
    x_pert = x_fid + 0.1 * np.random.normal(size=n_params)  
    starts.append(project_theta_np(x_mean_np + x_std_np * x_pert))

print(f"Running TensorFlow L-BFGS (normalised coords) from {len(starts)} starts...")
results = []
for k, th in enumerate(starts):
    chi2_start, _ = chi2_and_grad(th)
    res = lbfgs_minimise(th, include_gauss_prior=True)
    chi2_bf, _ = chi2_and_grad(res['theta'])
    logpost_bf, g_post = logpost_and_grad(res['theta'], include_gauss_prior=True)
    # gradient in normalised space (fairer convergence check)
    _, g_x = tfp.math.value_and_gradient(
        lambda z: neg_logpost_from_x(z, include_gauss_prior=True),
        tf.constant(x_from_theta(res['theta']), dtype=DTYPE),
    )
    g_x_norm = float(np.linalg.norm(g_x.numpy()))
    results.append({
        'start_idx': k,
        'converged': res['converged'],
        'failed': res['failed'],
        'nit': res['nit'],
        'theta': res['theta'],
        'chi2': chi2_bf,
        'logL': -0.5 * chi2_bf,
        'logpost': logpost_bf,
        'grad_logpost_norm': float(np.linalg.norm(g_post)),
        'grad_x_norm': g_x_norm,
        'objective': res['objective'],
        'chi2_start': chi2_start,
    })
    print(f"  start {k}: chi2 {chi2_start:.4g} -> {chi2_bf:.4g}, "
          #f"converged={res['converged']}, failed={res['failed']}, nit={res['nit']}, "
          f"logpost={logpost_bf:.4f}, ||∇_x||={g_x_norm:.3e}, ||∇_θ||={np.linalg.norm(g_post):.3e}")

best = max(results, key=lambda r: r['logpost'] if np.isfinite(r['logpost']) else -np.inf)
best_fit = params_array_to_dict(best['theta'])

print("\n" + "=" * 60)
print("Best-fit from TensorFlow L-BFGS (max log-posterior)")
print("=" * 60)
print(f"chi2             = {best['chi2']:.6f}")
print(f"logL             = {best['logL']:.6f}")
print(f"logpost          = {best['logpost']:.6f}")
print(f"||∇logpost||_θ   = {best['grad_logpost_norm']:.6e}")
print(f"||∇obj||_x       = {best['grad_x_norm']:.6e}  (normalised space)")
#print(f"converged        = {best['converged']}  failed={best['failed']}  "
#      f"(start {best['start_idx']}, nit={best['nit']})")
print("\nBest-fit parameters:")
for p in param_names:
    print(f"  {p:45s}  {best_fit[p]: .6g}   (fiducial {fiducials[p]: .6g})")


Running TensorFlow L-BFGS (normalised coords) from 8 starts...
  start 0: chi2 898.7 -> 0.04966, logpost=-0.0344, ||∇_x||=6.479e-01, ||∇_θ||=8.519e+01
  start 1: chi2 2890 -> 0.05225, logpost=-0.0316, ||∇_x||=3.221e-01, ||∇_θ||=2.707e+01
  start 2: chi2 124.8 -> 0.1291, logpost=-0.1169, ||∇_x||=1.240e+00, ||∇_θ||=1.654e+02
  start 3: chi2 183.5 -> 0.04493, logpost=-0.0307, ||∇_x||=3.212e-01, ||∇_θ||=5.633e+01
  start 4: chi2 868.6 -> 0.07825, logpost=-0.0616, ||∇_x||=7.473e-01, ||∇_θ||=9.900e+01
  start 5: chi2 165.1 -> 0.05361, logpost=-0.0405, ||∇_x||=1.971e+00, ||∇_θ||=9.244e+01
  start 6: chi2 84.06 -> 0.09592, logpost=-0.0804, ||∇_x||=1.104e+00, ||∇_θ||=1.446e+02
  start 7: chi2 1052 -> 1052, logpost=-526.0401, ||∇_x||=7.685e+03, ||∇_θ||=2.636e+05

Best-fit from TensorFlow L-BFGS (max log-posterior)
chi2             = 0.044932
logL             = -0.022466
logpost          = -0.030656
||∇logpost||_θ   = 5.633130e+01
||∇obj||_x       = 3.211523e-01  (normalised space)

Best-fit para